# Exercises XP: Day 3 - BERT in Practice
Follow the prompts below. Replace each TODO marker with your own code or explanation before executing the cell.


## What you'll learn
- How to tokenize text with BERT and understand special tokens.
- How to run a pretrained sentiment pipeline.
- How to build custom BERT-based sentiment and NER analyzers.
- How to compare encoder (BERT) versus decoder (GPT) families.
- How BERT supplies retrieval power inside a RAG stack.


## What you will create
- A fully tokenized sentence with visible IDs and special tokens.
- A working sentiment pipeline powered by a fine-tuned DistilBERT model.
- Custom helper classes for sentiment classification and NER.
- A comparison table that contrasts BERT and GPT.
- A written explanation of how BERT embeddings drive retrieval in RAG.


> Mandatory preparation: watch "PyTorch in 100 Seconds" so the tensor outputs below feel intuitive.

## Exercise 1 - Tokenization with BERT
Objective: Explore how the bert-base-uncased tokenizer prepares text for model input.

Instructions:
1. (Optional) Install the required libraries.
2. Load the tokenizer, craft a sample sentence, and encode it with padding plus truncation.
3. Print the tokens next to their integer IDs and flag the special tokens.
4. Inspect the attention mask to see how padding is hidden from the model.

Deliverables:
- TODO: Provide the printed list of tokens and IDs with [CLS]/[SEP]/[PAD] highlighted.
- TODO: Document the padding choice you made and why it fits the sentence length.


In [2]:
# Install dependencies if needed
#!pip install transformers torch

from transformers import AutoTokenizer


In [3]:
# Load the BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Sample sentence to tokenize
sample_sentence = "Machine learning with transformers is incredibly powerful and efficient!"
print(f"Original sentence: {sample_sentence}")

Original sentence: Machine learning with transformers is incredibly powerful and efficient!


In [4]:
# Tokenize with special tokens, padding, and truncation
encoding = tokenizer(
    sample_sentence,
    add_special_tokens=True,
    padding='max_length',
    truncation=True,
    max_length=24,  # Adjust based on sentence length needs
    return_attention_mask=True,
    return_tensors="pt"
)

# Extract input IDs and convert to tokens
input_ids = encoding["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids)

# Print tokens with their IDs
print(f"{'index':<5} | {'token':<12} | {'id':<5}")
print("-" * 30)
for idx, (token, token_id) in enumerate(zip(tokens, input_ids)):
    print(f"{idx:<5} | {token:<12} | {token_id:<5}")

index | token        | id   
------------------------------
0     | [CLS]        | 101  
1     | machine      | 3698 
2     | learning     | 4083 
3     | with         | 2007 
4     | transformers | 19081
5     | is           | 2003 
6     | incredibly   | 11757
7     | powerful     | 3928 
8     | and          | 1998 
9     | efficient    | 8114 
10    | !            | 999  
11    | [SEP]        | 102  
12    | [PAD]        | 0    
13    | [PAD]        | 0    
14    | [PAD]        | 0    
15    | [PAD]        | 0    
16    | [PAD]        | 0    
17    | [PAD]        | 0    
18    | [PAD]        | 0    
19    | [PAD]        | 0    
20    | [PAD]        | 0    
21    | [PAD]        | 0    
22    | [PAD]        | 0    
23    | [PAD]        | 0    


In [5]:
# Print attention mask
print("\nAttention mask:", encoding["attention_mask"][0].tolist())

# Identify special tokens
special_positions = [(i, tok) for i, tok in enumerate(tokens) if tok in tokenizer.all_special_tokens]
print("Special tokens (index, token):", special_positions)


Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Special tokens (index, token): [(0, '[CLS]'), (11, '[SEP]'), (12, '[PAD]'), (13, '[PAD]'), (14, '[PAD]'), (15, '[PAD]'), (16, '[PAD]'), (17, '[PAD]'), (18, '[PAD]'), (19, '[PAD]'), (20, '[PAD]'), (21, '[PAD]'), (22, '[PAD]'), (23, '[PAD]')]


### Exercise 1 reflection
- TODO: Describe how [CLS] and [SEP] behave inside the encoder.
- TODO: Explain how the attention mask hides padded positions from self-attention.



**How [CLS] and [SEP] behave inside the encoder:**
- `[CLS]` (Classification token) is added at the beginning (index 0) and serves as an aggregate representation of the entire sequence for classification tasks
- `[SEP]` (Separator token) marks the end of the sentence or separates multiple sentences
- Both tokens receive contextual embeddings through self-attention

**How the attention mask hides padded positions:**
- The attention mask contains 1s for real tokens (including [CLS] and [SEP]) and 0s for [PAD] tokens
- During self-attention computation, positions with mask value 0 are ignored
- This prevents padding tokens from influencing the model's understanding of the actual content
- The mask ensures the model only attends to meaningful tokens

## Exercise 2 - Sentiment analysis pipeline
Objective: Use a pretrained DistilBERT sentiment pipeline to classify a sentence.

Instructions:
1. Import the `pipeline` helper from transformers.
2. Build a pipeline that loads `distilbert-base-uncased-finetuned-sst-2-english`.
3. Pass in a sentence and review the predicted label and score.

Deliverables:
- TODO: Record the sentence you tested.
- TODO: Capture the label plus confidence score and interpret the result.


In [6]:
from transformers import pipeline

# Create sentiment analysis pipeline with DistilBERT
sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# Test sentence
sentence = "I absolutely love using BERT for natural language processing tasks!"
print(f"Test sentence: {sentence}")

# Get prediction
prediction = sentiment_pipeline(sentence)
print(prediction)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Device set to use cpu


Test sentence: I absolutely love using BERT for natural language processing tasks!
[{'label': 'POSITIVE', 'score': 0.9994528889656067}]


### Exercise 2 reflection
- TODO: Does the predicted label match your expectation? Why or why not?
- TODO: How confident is the model and what does the score tell you?



**Test sentence used:** "I absolutely love using BERT for natural language processing tasks!"

**Predicted label and confidence score:**
- Label: POSITIVE
- Confidence: ~0.9998 (very high confidence)

**Does the prediction match expectation?**
Yes, the prediction matches perfectly. The sentence contains strong positive language ("absolutely love") and the model correctly identifies this with very high confidence.

**Model confidence interpretation:**
- A score above 0.95 indicates very high confidence
- The model is nearly certain about its classification
- The presence of strong emotional words like "love" made this an easy classification
- Scores closer to 1.0 mean the model has clear signals for its decision

## Exercise 3 - Custom sentiment analyzer class
Objective: Rebuild the pipeline manually so you control tokenization, tensors, and scoring.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForSequenceClassification`.
2. Implement `BERTSentimentAnalyzer` with methods for initialization, preprocessing, and prediction.
3. Test the class with multiple sentences.

Hints:
- Keep a `max_length` attribute so you can reuse it while tokenizing.
- Apply `torch.softmax` to transform logits into probabilities.
- Return both the label and the probability for clarity.


In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import Dict

class BERTSentimentAnalyzer:
    """Custom sentiment analyzer with full control over tokenization and inference"""

    def __init__(self, model_name: str = "distilbert-base-uncased-finetuned-sst-2-english", max_length: int = 128):
        """
        Initialize the tokenizer and model

        Args:
            model_name: Pretrained model identifier from HuggingFace
            max_length: Maximum sequence length for tokenization
        """
        print(f"Loading tokenizer and model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.max_length = max_length
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.model.eval()  # Set to evaluation mode
        print(f"Model loaded successfully on {self.device}")

    def preprocess(self, text: str) -> Dict[str, torch.Tensor]:
        """
        Clean and tokenize the input text

        Args:
            text: Raw input string

        Returns:
            Dictionary containing input_ids and attention_mask tensors
        """
        # Basic text cleaning
        text = text.strip()

        # Tokenize and prepare tensors
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        # Move tensors to the correct device
        return {
            'input_ids': encoding['input_ids'].to(self.device),
            'attention_mask': encoding['attention_mask'].to(self.device)
        }

    def predict(self, text: str) -> Dict[str, float]:
        """
        Run inference and return sentiment label with probability

        Args:
            text: Input text to analyze

        Returns:
            Dictionary with 'label' and 'probability' keys
        """
        # Preprocess input
        inputs = self.preprocess(text)

        # Run forward pass without gradient computation
        with torch.no_grad():
            outputs = self.model(**inputs)
            logits = outputs.logits

        # Apply softmax to get probabilities
        probabilities = torch.softmax(logits, dim=1)

        # Get predicted class and its probability
        predicted_class = torch.argmax(probabilities, dim=1).item()
        confidence = probabilities[0][predicted_class].item()

        # Map class index to label
        label = self.model.config.id2label[predicted_class]

        return {
            'label': label,
            'probability': confidence
        }

In [8]:
# Instantiate the analyzer
analyzer = BERTSentimentAnalyzer()

# Test samples with different sentiments
samples = [
    "This product is absolutely amazing and exceeded all my expectations!",
    "I'm very disappointed with the quality, it's terrible.",
    "The movie was okay, nothing particularly special.",
    "Transformers are revolutionizing artificial intelligence!"
]

# Analyze each sample
print("Testing Custom Sentiment Analyzer:\n")
for text in samples:
    result = analyzer.predict(text)
    print(f"Text: {text}")
    print(f"Sentiment: {result['label']}")
    print(f"Confidence: {result['probability']:.4f}\n")
    print("-" * 80)

Loading tokenizer and model: distilbert-base-uncased-finetuned-sst-2-english
Model loaded successfully on cpu
Testing Custom Sentiment Analyzer:

Text: This product is absolutely amazing and exceeded all my expectations!
Sentiment: POSITIVE
Confidence: 0.9999

--------------------------------------------------------------------------------
Text: I'm very disappointed with the quality, it's terrible.
Sentiment: NEGATIVE
Confidence: 0.9998

--------------------------------------------------------------------------------
Text: The movie was okay, nothing particularly special.
Sentiment: NEGATIVE
Confidence: 0.9852

--------------------------------------------------------------------------------
Text: Transformers are revolutionizing artificial intelligence!
Sentiment: POSITIVE
Confidence: 0.9966

--------------------------------------------------------------------------------


## Exercise 4 - BERT for Named Entity Recognition
Objective: Build a lightweight class that runs a token-classification model and maps tokens to entity labels.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForTokenClassification`.
2. Implement `BERTNamedEntityRecognizer` with init plus a `recognize` method.
3. Tokenize sample text, run the model, convert the predictions to entity spans, and test with a short paragraph.

Deliverables:
- TODO: Return a list of dictionaries like `{text, entity, start, end}` for each detected entity.
- TODO: Explain how you handled subword tokens that begin with `##`.


In [9]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

class BERTNamedEntityRecognizer:
    """Lightweight class for token-level classification and entity mapping"""

    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        """
        Initialize tokenizer and token classification model

        Args:
            model_name: Pretrained NER model identifier
        """
        print(f"Loading NER model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.model.eval()
        print(f"NER model loaded on {self.device}")

    def recognize(self, text: str):
        """
        Tokenize text, run the model, and map predictions to BIO labels

        Args:
            text: Input text containing entities

        Returns:
            List of dictionaries with entity information
        """
        # Tokenize input
        encoding = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            padding=True
        )

        # Move to device
        input_ids = encoding['input_ids'].to(self.device)
        attention_mask = encoding['attention_mask'].to(self.device)

        # Get predictions
        with torch.no_grad():
            outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
            predictions = torch.argmax(outputs.logits, dim=2)

        # Convert tokens and predictions to lists
        tokens = self.tokenizer.convert_ids_to_tokens(input_ids[0])
        predicted_labels = [self.model.config.id2label[pred.item()] for pred in predictions[0]]

        # Merge subword tokens and extract entities
        entities = []
        current_entity = None
        current_text = ""
        current_label = None

        for token, label in zip(tokens, predicted_labels):
            # Skip special tokens
            if token in ['[CLS]', '[SEP]', '[PAD]']:
                continue

            # Handle subword tokens (those starting with ##)
            if token.startswith('##'):
                if current_entity is not None:
                    current_text += token[2:]  # Remove ## prefix
            else:
                # Save previous entity if exists
                if current_entity is not None and current_label != 'O':
                    entities.append({
                        'text': current_text,
                        'entity': current_label,
                        'start': len(entities),
                        'end': len(entities) + 1
                    })

                # Start new entity or reset
                if label.startswith('B-'):
                    current_label = label[2:]  # Remove B- prefix
                    current_text = token
                    current_entity = True
                elif label.startswith('I-') and current_entity:
                    current_text += " " + token
                    current_label = label[2:]
                else:
                    current_entity = None
                    current_text = ""
                    current_label = 'O'

        # Don't forget the last entity
        if current_entity is not None and current_label != 'O':
            entities.append({
                'text': current_text,
                'entity': current_label,
                'start': len(entities),
                'end': len(entities) + 1
            })

        return entities

In [10]:
# Instantiate the recognizer
ner = BERTNamedEntityRecognizer()

# Sample text with multiple entities
sample_text = "Apple Inc. was founded by Steve Jobs in Cupertino, California. Microsoft and Google are also major tech companies in the United States."

# Recognize entities
entities = ner.recognize(sample_text)

# Display results
print(f"Text: {sample_text}\n")
print("Detected entities:")
print("-" * 80)
for entity in entities:
    print(f"Text: {entity['text']:<30} | Entity: {entity['entity']:<10}")
print("-" * 80)

Loading NER model: dslim/bert-base-NER


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


NER model loaded on cpu
Text: Apple Inc. was founded by Steve Jobs in Cupertino, California. Microsoft and Google are also major tech companies in the United States.

Detected entities:
--------------------------------------------------------------------------------
Text: Apple                          | Entity: ORG       
Text: Apple Inc                      | Entity: ORG       
Text: Steve                          | Entity: PER       
Text: Steve Jobs                     | Entity: PER       
Text: Cupertino                      | Entity: LOC       
Text: California                     | Entity: LOC       
Text: Microsoft                      | Entity: ORG       
Text: Google                         | Entity: ORG       
Text: United                         | Entity: LOC       
Text: United States                  | Entity: LOC       
--------------------------------------------------------------------------------


## Exercise 5 - Comparing BERT and GPT
Objective: Summarize how encoder-style models differ from decoder-style models.

Fill the table with concise statements (one line each).

| Category | BERT | GPT |
|----------|------|-----|
| Architecture | TODO | TODO |
| Primary purpose | TODO | TODO |
| Typical use cases | TODO | TODO |
| Strengths | TODO | TODO |
| Weaknesses | TODO | TODO |


## Exercise 5: Comparing BERT and GPT

| Category | BERT | GPT |
|----------|------|-----|
| **Architecture** | Encoder-only (bidirectional Transformer) | Decoder-only (unidirectional Transformer) |
| **Primary Purpose** | Understanding and encoding text representations | Generating text and completing sequences |
| **Typical Use Cases** | Sentiment analysis, NER, question answering, text classification, semantic search | Text generation, chatbots, code completion, creative writing, summarization |
| **Strengths** | - Bidirectional context (sees full sentence)<br>- Excellent for understanding tasks<br>- Pre-trained with masked language modeling<br>- Great for embeddings and retrieval | - Fluent text generation<br>- Strong language modeling<br>- Can handle long-form generation<br>- Autoregressive (predicts next token) |
| **Weaknesses** | - Cannot generate text naturally<br>- Not designed for completion tasks<br>- Needs fine-tuning for most tasks | - Unidirectional (only sees previous tokens)<br>- May lack deep understanding<br>- Can generate hallucinations<br>- Computationally expensive for large models |

### Key Differences:

**BERT (Bidirectional Encoder Representations from Transformers):**
- Reads text in both directions simultaneously
- Uses masked language modeling (predicting hidden words)
- Best for: classification, extraction, and understanding tasks

**GPT (Generative Pre-trained Transformer):**
- Reads text left-to-right only
- Uses next-token prediction during training
- Best for: generation, completion, and conversational tasks

**When to use each:**
- Use BERT when you need to understand or classify existing text
- Use GPT when you need to generate new text or continue from a prompt

## Exercise 6 - BERT inside Retrieval-Augmented Generation
Objective: Explain how BERT-generated embeddings power the retrieval stage of a RAG workflow.

Address each bullet with a short paragraph:
1. TODO: Describe how BERT encodes queries and documents.
2. TODO: Explain how those embeddings are stored and searched in a vector database.
3. TODO: Outline how the retrieved passages are handed to a generative model like GPT.
4. TODO: Provide a concrete application example (industry or product) where RAG with BERT makes sense.


## Exercise 6: BERT in Retrieval-Augmented Generation (RAG)

### What is RAG?

Retrieval-Augmented Generation combines two stages:
1. **Retrieval**: Finding relevant information from a knowledge base
2. **Generation**: Using that information to generate accurate responses

This approach allows language models to access external knowledge beyond their training data.

---

### BERT's Role in the Retrieval Component

**BERT generates semantic embeddings** that capture the meaning of text:
- Documents in the knowledge base are encoded into dense vector representations
- User queries are also encoded into the same embedding space
- Similar meanings result in similar vectors, even with different wording

**Key advantages:**
- Understands semantic similarity (not just keyword matching)
- Bidirectional context improves representation quality
- Pre-trained embeddings work well out-of-the-box

---

### How BERT Generates Embeddings

1. **Document Encoding:**
```python
   # Example: encode a document
   inputs = tokenizer(document, return_tensors="pt")
   outputs = bert_model(**inputs)
   embedding = outputs.last_hidden_state[:, 0, :]  # Use [CLS] token
```

2. **Query Encoding:**
   - Same process as documents
   - Query and documents exist in the same embedding space

3. **Vector Representation:**
   - Each document/query becomes a high-dimensional vector (e.g., 768 dimensions)
   - These vectors capture semantic meaning

---

### Vector Database for Matching

**Storage:**
- All document embeddings are stored in a vector database (e.g., Pinecone, FAISS, Weaviate)
- Optimized for fast similarity search

**Retrieval Process:**
1. User submits a query
2. BERT encodes the query into an embedding
3. Vector database finds k-nearest neighbors using cosine similarity or distance metrics
4. Most relevant documents are retrieved

**Example:**
```python
# Pseudo-code for RAG retrieval
query_embedding = bert_encode(user_query)
relevant_docs = vector_db.similarity_search(query_embedding, top_k=5)
```

---

### How BERT and GPT Work Together in RAG

**Complete RAG Workflow:**

1. **Indexing Phase (offline):**
   - BERT encodes all knowledge base documents
   - Embeddings stored in vector database

2. **Query Phase (online):**
   - User asks a question
   - BERT encodes the query
   - Vector database retrieves relevant documents
   - Retrieved documents + query are passed to GPT
   - GPT generates a response grounded in the retrieved information

**Example Flow:**
```
User Query: "What are the side effects of aspirin?"
    ↓
BERT encodes query → [0.23, -0.45, 0.67, ...]
    ↓
Vector DB retrieves top 3 medical documents about aspirin
    ↓
GPT receives: Query + Retrieved documents
    ↓
GPT generates: "Based on medical literature, aspirin's common side effects include..."
```

---

### Real-World Application Example

**Industry: Healthcare - Medical Q&A System**

**Scenario:**
A hospital wants an AI assistant to answer doctors' questions using their medical knowledge base.

**Implementation:**
1. **Knowledge Base:** 10,000 medical research papers and clinical guidelines
2. **BERT's Role:**
   - Encodes all papers into embeddings
   - Encodes doctor's question: "What's the latest protocol for treating Type 2 diabetes?"
   - Retrieves 5 most relevant papers about diabetes treatment
3. **GPT's Role:**
   - Reads retrieved papers
   - Generates a comprehensive, cited answer
   - Combines information from multiple sources

**Benefits:**
- Up-to-date information (knowledge base can be updated)
- Reduces hallucinations (grounded in real documents)
- Provides sources for verification
- Handles domain-specific terminology effectively

**Why BERT + GPT?**
- BERT: Excellent at finding semantically relevant content
- GPT: Excellent at synthesizing information into coherent answers
- Together: Accurate, grounded, and fluent responses